# H&M Recommendations - Part 2b: ALS Collaborative Filtering

This notebook adds ALS (Alternating Least Squares) matrix factorization as an additional retrieval strategy on top of the existing rule-based candidates, and computes CF similarity scores to use as reranker features.

Two outputs:
1. **ALS candidates** - top-N items per user based on learned user/item embeddings, saved to `als_candidates.parquet`
2. **CF score features** - item2item CF similarity scores between candidate and user history, saved to `cf_features.parquet`

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install implicit -q

import pandas as pd
import numpy as np
import scipy.sparse as sparse
import implicit
import os
from datetime import timedelta

DATA_DIR = '/content/drive/MyDrive/HM Fashion/processed/'
CAND_DIR = '/content/drive/MyDrive/HM Fashion/candidates/'
os.makedirs(CAND_DIR, exist_ok=True)

print(f'implicit version: {implicit.__version__}')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
implicit version: 0.7.2


## 2. Load Data

I use the same train/test split as the rest of the pipeline - last 7 days as test.

In [ ]:
train     = pd.read_parquet(DATA_DIR + 'train.parquet')
test      = pd.read_parquet(DATA_DIR + 'test.parquet')
customers = pd.read_csv(DATA_DIR + 'customers.csv')
articles  = pd.read_csv(DATA_DIR + 'articles.csv', dtype={'article_id': str})

ground_truth = test.groupby('customer_id')['article_id'].apply(set).reset_index()
ground_truth.columns = ['customer_id', 'purchased']

split_date = pd.to_datetime(train['t_dat'].max())
test_users = test['customer_id'].unique()

print(f'Train         : {len(train):,}')
print(f'Test users    : {len(test_users):,}')
print(f'Unique items  : {train.article_id.nunique():,}')
print(f'Unique users  : {train.customer_id.nunique():,}')


Train         : 31,548,013
Test users    : 68,984
Unique items  : 103,880
Unique users  : 1,356,709


## 3. Build User-Item Matrix

ALS works on an implicit feedback matrix where each entry represents how many times a user bought an item. I use the last 6 weeks of train data (recently active items) to keep the matrix focused on current preferences.

In [ ]:
# use last 6 weeks - same window as rule-based retrieval
six_weeks_ago   = split_date - pd.Timedelta(weeks=6)
train_recent    = train[train['t_dat'] >= six_weeks_ago].copy()
recently_active = set(train_recent['article_id'].unique())

print(f'Recently active items (6w): {len(recently_active):,}')
print(f'Train recent transactions : {len(train_recent):,}')


Recently active items (6w): 33,146
Train recent transactions : 1,668,970


In [ ]:
# encode customer_id and article_id as integers
all_users_list  = train_recent['customer_id'].unique().tolist()
all_items_list  = train_recent['article_id'].unique().tolist()

user2idx = {u: i for i, u in enumerate(all_users_list)}
item2idx = {a: i for i, a in enumerate(all_items_list)}
idx2user = {i: u for u, i in user2idx.items()}
idx2item = {i: a for a, i in item2idx.items()}

n_users = len(all_users_list)
n_items = len(all_items_list)

print(f'Matrix size: {n_users:,} users x {n_items:,} items')


Matrix size: 319,129 users x 33,146 items


In [ ]:
# build sparse user-item matrix (purchase counts as implicit feedback)
user_indices = train_recent['customer_id'].map(user2idx).values
item_indices = train_recent['article_id'].map(item2idx).values
counts       = np.ones(len(train_recent), dtype=np.float32)

# user-item matrix: rows=users, cols=items
user_item_matrix = sparse.csr_matrix(
    (counts, (user_indices, item_indices)),
    shape=(n_users, n_items)
)

# item-user matrix needed by implicit library
item_user_matrix = user_item_matrix.T.tocsr()

print(f'User-item matrix: {user_item_matrix.shape}')
print(f'Density         : {user_item_matrix.nnz / (n_users * n_items):.6%}')


User-item matrix: (319129, 33146)
Density         : 0.013943%


## 4. Train ALS Model

ALS learns k-dimensional latent vectors for each user and item. The dot product of user and item vectors gives a relevance score. Key parameters:
- `factors` - embedding dimension (higher = more expressive but slower)
- `iterations` - training iterations (20 is usually enough for implicit feedback)
- `regularization` - prevents overfitting

In [ ]:
# train ALS model
als_model = implicit.als.AlternatingLeastSquares(
    factors        = 64,
    iterations     = 20,
    regularization = 0.01,
    random_state   = 42,
    use_gpu        = implicit.gpu.HAS_CUDA,
)

print(f'Using GPU: {implicit.gpu.HAS_CUDA}')
print('Training ALS...')
# train ALS - fit on user_item_matrix directly
als_model.fit(user_item_matrix)
print('Done.')


Using GPU: True
Training ALS...


  0%|          | 0/20 [00:00<?, ?it/s]

Done.


## 5. Evaluate ALS Recall

Before generating candidates let me check the recall of ALS recommendations. I'll compare against the rule-based baseline.

In [ ]:
def eval_recall(candidates_df, ground_truth):
    merged = candidates_df.merge(ground_truth, on='customer_id', how='inner')
    recall = merged.apply(
        lambda row: len(set(row['article_id']) & row['purchased']) / len(row['purchased']),
        axis=1).mean()
    avg_hits = merged.apply(
        lambda row: len(set(row['article_id']) & row['purchased']), axis=1).mean()
    zero_hit = merged.apply(
        lambda row: len(set(row['article_id']) & row['purchased']) == 0, axis=1).mean()
    return recall, avg_hits, zero_hit


In [ ]:
# generate ALS candidates for test users
N_CANDIDATES = 50

test_users_in_matrix = [u for u in test_users if u in user2idx]
test_users_cold      = [u for u in test_users if u not in user2idx]

print(f'Test users in ALS matrix : {len(test_users_in_matrix):,}')
print(f'Test users cold (no ALS) : {len(test_users_cold):,}')
print(f'Generating {N_CANDIDATES} ALS candidates per user...')

als_results = []
for i, user_id in enumerate(test_users_in_matrix):
    u_idx = user2idx[user_id]
    # use recalculate_user=True since model was trained on item_user_matrix
    ids, scores = als_model.recommend(
        u_idx,
        user_item_matrix[u_idx],
        N=N_CANDIDATES,
        filter_already_liked_items=True,
        recalculate_user=True  # ← recalculates user vector from interactions
    )
    items = [idx2item[int(idx)] for idx in ids if int(idx) in idx2item]
    als_results.append({'customer_id': user_id, 'article_id': items})

    if i % 5000 == 0:
        print(f'  {i:,} / {len(test_users_in_matrix):,}')

als_candidates = pd.DataFrame(als_results)
print(f'Generated candidates for {len(als_candidates):,} users')
print(f'Avg candidates per user  : {als_candidates["article_id"].apply(len).mean():.1f}')

Test users in ALS matrix : 38,028
Test users cold (no ALS) : 30,956
Generating 50 ALS candidates per user...
  0 / 38,028
  5,000 / 38,028
  10,000 / 38,028
  15,000 / 38,028
  20,000 / 38,028
  25,000 / 38,028
  30,000 / 38,028
  35,000 / 38,028
Generated candidates for 38,028 users
Avg candidates per user  : 50.0


In [ ]:
# evaluate ALS recall
recall, avg_hits, zero_hit = eval_recall(als_candidates, ground_truth)
print(f'ALS Recall@{N_CANDIDATES}    : {recall:.4f}')
print(f'ALS Avg hits         : {avg_hits:.3f}')
print(f'ALS Zero-hit users   : {zero_hit:.1%}')
print(f'ALS Coverage         : {len(als_candidates)/len(test_users):.1%} of test users')


ALS Recall@50    : 0.0570
ALS Avg hits         : 0.154
ALS Zero-hit users   : 87.1%
ALS Coverage         : 55.1% of test users


## 6. Combine ALS + Rule-Based Candidates

The real value of ALS is in combination with rule-based candidates - they cover different types of users and find different relevant items. I'll combine both and measure the recall improvement.

In [ ]:
# load rule-based candidates
rule_candidates = pd.read_parquet(CAND_DIR + 'rule_candidates.parquet')
print(f'Rule candidates: {len(rule_candidates):,} users, avg {rule_candidates["article_id"].apply(len).mean():.1f} per user')


Rule candidates: 68,984 users, avg 300.0 per user


In [ ]:
# combine rule + ALS candidates
def combine_candidates(rule_df, als_df):
    combined = rule_df.merge(als_df, on='customer_id', how='left',
                              suffixes=('_rule', '_als'))
    results = []
    for _, row in combined.iterrows():
        rule_items = list(row['article_id_rule']) if isinstance(row['article_id_rule'], (list, np.ndarray)) else []
        als_items  = list(row['article_id_als'])  if isinstance(row['article_id_als'],  (list, np.ndarray)) else []

        seen   = set()
        merged = []
        for item in rule_items + als_items:
            if item not in seen:
                seen.add(item)
                merged.append(item)
        results.append({'customer_id': row['customer_id'], 'article_id': merged})
    return pd.DataFrame(results)

combined_candidates = combine_candidates(rule_candidates, als_candidates)
print(f'Combined candidates: {len(combined_candidates):,} users, avg {combined_candidates["article_id"].apply(len).mean():.1f} per user')

Combined candidates: 68,984 users, avg 314.9 per user


In [ ]:
# compare recall
rule_recall,   rule_hits,   rule_zero   = eval_recall(rule_candidates,   ground_truth)
als_recall,    als_hits,    als_zero    = eval_recall(als_candidates,    ground_truth)
combo_recall,  combo_hits,  combo_zero  = eval_recall(combined_candidates, ground_truth)

print(f'{"Strategy":<35} {"Recall":>8} {"Avg hits":>10} {"Zero-hit%":>10}')
print('-' * 67)
print(f'{"Rule-based only (300 cands)":<35} {rule_recall:>8.4f} {rule_hits:>10.3f} {rule_zero:>9.1%}')
print(f'{f"ALS only ({N_CANDIDATES} cands)":<35} {als_recall:>8.4f} {als_hits:>10.3f} {als_zero:>9.1%}')
print(f'{"Rule + ALS combined":<35} {combo_recall:>8.4f} {combo_hits:>10.3f} {combo_zero:>9.1%}')

# unique hits from ALS not in rule-based
merged_both = rule_candidates.merge(als_candidates, on='customer_id', how='inner',
                                     suffixes=('_rule', '_als'))
merged_both = merged_both.merge(ground_truth, on='customer_id', how='inner')
merged_both['unique_als_hits'] = merged_both.apply(
    lambda row: len(set(row['article_id_als']) & row['purchased'] -
                    set(row['article_id_rule'])), axis=1)
print(f'\nAvg unique hits from ALS not in rule-based : {merged_both["unique_als_hits"].mean():.4f}')
print(f'Users where ALS finds something new        : {(merged_both["unique_als_hits"] > 0).mean():.1%}')


Strategy                              Recall   Avg hits  Zero-hit%
-------------------------------------------------------------------
Rule-based only (300 cands)           0.2747      0.787     51.7%
ALS only (50 cands)                   0.0570      0.154     87.1%
Rule + ALS combined                   0.2818      0.808     50.8%

Avg unique hits from ALS not in rule-based : 0.0389
Users where ALS finds something new        : 3.6%


## 7. CF Score Features for Reranker

Beyond candidate generation, ALS item vectors can be used to compute similarity scores between candidate items and user purchase history. This is the item2item CF similarity score used as a reranker feature in the 1st place solution.

Two features:
- `als_user_score` - direct ALS score for this user-item pair (how much the model thinks this user wants this item)
- `als_item_cf_score` - average cosine similarity between candidate item and user's purchased items (item2item CF)

In [ ]:
from numpy.linalg import norm

# get item factors (embeddings)
item_factors = als_model.item_factors  # shape: (n_items, factors)
user_factors = als_model.user_factors  # shape: (n_users, factors)

print(f'Item factors shape: {item_factors.shape}')
print(f'User factors shape: {user_factors.shape}')


Item factors shape: (33146, 64)
User factors shape: (319129, 64)


In [ ]:
# convert GPU matrices to numpy
item_factors = als_model.item_factors
user_factors = als_model.user_factors

if hasattr(item_factors, 'to_numpy'):
    item_factors = item_factors.to_numpy()
    user_factors = user_factors.to_numpy()

print(f'Item factors shape: {item_factors.shape}')
print(f'User factors shape: {user_factors.shape}')

# precompute normalized item factors for cosine similarity
item_norms          = norm(item_factors, axis=1, keepdims=True)
item_norms[item_norms == 0] = 1e-8
item_factors_normed = item_factors / item_norms

# user purchase history for CF score computation
user_history = (train_recent
                .sort_values('t_dat', ascending=False)
                .groupby('customer_id')['article_id']
                .apply(lambda x: list(dict.fromkeys(x))[:20])
                .to_dict())

print(f'Users with history: {len(user_history):,}')

Item factors shape: (33146, 64)
User factors shape: (319129, 64)
Users with history: 319,129


In [ ]:
# load combined candidates to compute CF features
# explode candidates into (customer_id, article_id) pairs
pool = combined_candidates.explode('article_id').reset_index(drop=True)
pool = pool[pool['article_id'].notna()].copy()

print(f'Pool size: {len(pool):,} pairs')


Pool size: 21,719,837 pairs


In [ ]:
# compute CF features for each (user, candidate) pair
print('Computing CF features...')

als_user_scores    = []
als_item_cf_scores = []

for i, row in pool.iterrows():
    user_id    = row['customer_id']
    article_id = row['article_id']

    # ── als_user_score: direct ALS prediction score ────────────────────
    if user_id in user2idx and article_id in item2idx:
        u_idx  = user2idx[user_id]
        i_idx  = item2idx[article_id]
        score  = float(np.dot(user_factors[u_idx], item_factors[i_idx]))
    else:
        score = 0.0
    als_user_scores.append(score)

    # ── als_item_cf_score: avg cosine sim to user history ──────────────
    history = user_history.get(user_id, [])
    if article_id in item2idx and history:
        cand_vec    = item_factors_normed[item2idx[article_id]]
        hist_vecs   = np.array([
            item_factors_normed[item2idx[h]]
            for h in history if h in item2idx
        ])
        if len(hist_vecs) > 0:
            cf_score = float(np.mean(hist_vecs @ cand_vec.T))
        else:
            cf_score = 0.0
    else:
        cf_score = 0.0
    als_item_cf_scores.append(cf_score)

    if i % 500000 == 0:
        print(f'  {i:,} / {len(pool):,}')

pool['als_user_score']    = als_user_scores
pool['als_item_cf_score'] = als_item_cf_scores

print('Done.')
print(f'als_user_score mean    : {pool["als_user_score"].mean():.4f}')
print(f'als_item_cf_score mean : {pool["als_item_cf_score"].mean():.4f}')


Computing CF features...
  0 / 21,719,837
  500,000 / 21,719,837
  1,000,000 / 21,719,837
  1,500,000 / 21,719,837
  2,000,000 / 21,719,837
  2,500,000 / 21,719,837
  3,000,000 / 21,719,837
  3,500,000 / 21,719,837
  4,000,000 / 21,719,837
  4,500,000 / 21,719,837
  5,000,000 / 21,719,837
  5,500,000 / 21,719,837
  6,000,000 / 21,719,837
  6,500,000 / 21,719,837
  7,000,000 / 21,719,837
  7,500,000 / 21,719,837
  8,000,000 / 21,719,837
  8,500,000 / 21,719,837
  9,000,000 / 21,719,837
  9,500,000 / 21,719,837
  10,000,000 / 21,719,837
  10,500,000 / 21,719,837
  11,000,000 / 21,719,837
  11,500,000 / 21,719,837
  12,000,000 / 21,719,837
  12,500,000 / 21,719,837
  13,000,000 / 21,719,837
  13,500,000 / 21,719,837
  14,000,000 / 21,719,837
  14,500,000 / 21,719,837
  15,000,000 / 21,719,837
  15,500,000 / 21,719,837
  16,000,000 / 21,719,837
  16,500,000 / 21,719,837
  17,000,000 / 21,719,837
  17,500,000 / 21,719,837
  18,000,000 / 21,719,837
  18,500,000 / 21,719,837
  19,000,000 / 21

## 8. Quick Evaluation of CF Features

Let me check if the CF scores are meaningful - items with higher scores should have higher purchase rates.

In [ ]:
# check if CF scores correlate with purchases
gt_exploded = ground_truth.explode('purchased').copy()
gt_exploded['is_purchased'] = 1
gt_exploded = gt_exploded.rename(columns={'purchased': 'article_id'})

pool_with_label = pool.merge(
    gt_exploded[['customer_id', 'article_id', 'is_purchased']],
    on=['customer_id', 'article_id'], how='left')
pool_with_label['is_purchased'] = pool_with_label['is_purchased'].fillna(0).astype(int)

# compare mean scores for purchased vs not purchased
print('=== als_user_score ===')
print(pool_with_label.groupby('is_purchased')['als_user_score'].mean().to_string())
print()
print('=== als_item_cf_score ===')
print(pool_with_label.groupby('is_purchased')['als_item_cf_score'].mean().to_string())

=== als_user_score ===
is_purchased
0    0.003600
1    0.017032

=== als_item_cf_score ===
is_purchased
0    0.080557
1    0.132345


## 9. Save Outputs

Save ALS candidates and CF features separately so they can be used in the reranker notebook.

In [ ]:
# save ALS candidates
combined_candidates.to_parquet(CAND_DIR + 'als_rule_candidates.parquet', index=False)
print(f'Saved als_rule_candidates.parquet ({len(combined_candidates):,} users)')

# save CF features (customer_id, article_id, als_user_score, als_item_cf_score)
cf_features = pool[['customer_id', 'article_id', 'als_user_score', 'als_item_cf_score']]
cf_features.to_parquet(CAND_DIR + 'cf_features.parquet', index=False)
print(f'Saved cf_features.parquet ({len(cf_features):,} pairs)')
